# Smart City MLOps: Nova Stad Verkeersmanagementsysteem
**Auteurs:** Nika de Vries, Thida Churam, Soner Yigitdol

Dit notebook bevat de Machine Learning pipeline voor het Nova Stad project, uitgevoerd in een lokale VS Code / GitHub omgeving. Het project bestaat uit twee componenten:
1. **Cloud Model (Tabeldata):** Verkeersvolume voorspellen m.b.v. PySpark en Random Forest.
2. **Edge Model (Beeldata):** Realtime objectdetectie op kruispunten met YOLOv8 (geëxporteerd naar TFLite).

We hanteren strikte MLOps-standaarden: alle experimenten worden gelogd via een lokale MLflow tracking server, en de PySpark-code is schaalbaar ontworpen (`local[*]`) zodat deze direct naar een cloud-cluster gemigreerd kan worden.

In [ ]:
# Installeer de benodigde packages (run dit 1x in VS Code of via je requirements.txt)
%pip install pyspark ultralytics mlflow evidently pandas numpy
!pip install -U ml_dtypes

  Using cached protobuf-6.33.6-cp39-abi3-macosx_10_9_universal2.whl.metadata (593 bytes)
  Using cached statsmodels-0.14.6-cp310-cp310-macosx_11_0_arm64.whl.metadata (9.5 kB)
  Using cached distro-1.9.0-py3-none-any.whl.metadata (6.8 kB)
  Using cached sniffio-1.3.1-py3-none-any.whl.metadata (3.9 kB)
  Using cached patsy-1.0.2-py2.py3-none-any.whl.metadata (3.6 kB)
  Using cached mypy_extensions-1.1.0-py3-none-any.whl.metadata (1.1 kB)
Using cached protobuf-6.33.6-cp39-abi3-macosx_10_9_universal2.whl (427 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 18.5 MB/s  0:00:00m0:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.1/19.1 MB 23.7 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 581.8/581.8 kB 17.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 21.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 22.8 MB/s  0:00:00
Using cached sniffio-1.3.1-py3-none-any.whl (10 kB)
Using cached statsmodels-0.14.6

In [10]:
import os

# 1. Cloud Data (Tabel) Paden
TRAIN_DATA_PATH = "/Users/thidaratchuram/Documents/GitHub/MLops-City/Metro_Interstate_Traffic_Volume_train.csv"
TEST_DATA_PATH  = "/Users/thidaratchuram/Documents/GitHub/MLops-City/Metro_Interstate_Traffic_Volume_test.csv"

# 2. Edge Data (Beelden) Paden
# Let op: we gebruiken /*.jpg voor de PySpark inlezing
IMG_TRAIN_PATH  = "/Users/thidaratchuram/Documents/GitHub/MLops-City/BDD/images/train/*.jpg"
LABEL_TRAIN_PATH = "/Users/thidaratchuram/Documents/GitHub/MLops-City/BDD/labels/train/*.txt"

# 3. MLflow Tracking Setup
# Dit maakt een map genaamd 'mlruns' aan in je huidige werkmap
MLFLOW_TRACKING_URI = "file://" + os.path.abspath("./mlruns")

In [11]:
import time
import mlflow
import numpy as np
import pandas as pd

# PySpark Imports
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, trim, regexp_replace, to_timestamp, year, month, dayofmonth, hour, dayofweek
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
from pyspark.ml import Pipeline

# YOLO Imports
from ultralytics import YOLO

# Start een lokale Spark sessie met alle CPU cores ("local[*]")
spark = SparkSession.builder \
    .appName("NovaStad_Pipeline_Local") \
    .master("local[*]") \
    .config("spark.driver.port", "random-free-port")

print("Lokale Spark Sessie gestart")

Lokale Spark Sessie gestart


## 1. Cloud Pipeline: Data Ingestion & Preprocessing
We definiëren vooraf een expliciet schema (`StructType`) voor de CSV-data. Dit is een best practice voor PySpark om schaalbaarheid te garanderen. We vullen missende temperatuurwaarden op met de mediaan, en ontbrekende neerslag (`rain_1h`, `snow_1h`) met `0.0`.

In [12]:
import os

print(TRAIN_DATA_PATH)
print(os.path.exists(TRAIN_DATA_PATH))


/Users/thidaratchuram/Documents/GitHub/MLops-City/Metro_Interstate_Traffic_Volume_train.csv
True


In [13]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType
from pyspark.sql.functions import (
    col, year, month, dayofmonth, hour, dayofweek,
    trim, regexp_replace, try_to_timestamp, lit,   # <-- added lit
)

metro_schema = StructType([
    StructField("_c0",                 IntegerType(), True),   # pandas index — dropped in cleaning
    StructField("holiday",             StringType(),  True),
    StructField("temp",                DoubleType(),  True),
    StructField("rain_1h",             DoubleType(),  True),
    StructField("snow_1h",             DoubleType(),  True),
    StructField("clouds_all",          IntegerType(), True),
    StructField("weather_main",        StringType(),  True),
    StructField("weather_description", StringType(),  True),
    StructField("date_time",           StringType(),  True),   # parsed in clean_cloud_data
    StructField("year",                IntegerType(), True),   # redundant — re-derived
    StructField("month",               IntegerType(), True),   # redundant
    StructField("day",                 IntegerType(), True),   # redundant
    StructField("hour",                StringType(),  True),   # raw is "09:00" — dropped + re-derived
    StructField("traffic_volume",      IntegerType(), True),
])

JUNK_COLS = ["_c0", "year", "month", "day", "hour"]   # index + pre-split date parts

def clean_cloud_data(df):
    # 1. Drop shipped index + redundant date parts FIRST (else _c0 breaks dropDuplicates).
    df = df.drop(*JUNK_COLS)
    df = df.dropDuplicates()

    # 2. Parse timestamp (format must be wrapped in lit), discard unparseable rows.
    df = df.withColumn("date_time", try_to_timestamp(col("date_time"), lit("yyyy-MM-dd HH:mm:ss")))
    df = df.filter(col("date_time").isNotNull())
    if df.rdd.isEmpty():
        raise ValueError("Empty after timestamp parsing — check the date_time format")

    # 3. Re-derive clean date features.
    df = (df
        .withColumn("year",      year(col("date_time")))
        .withColumn("month",     month(col("date_time")))
        .withColumn("day",       dayofmonth(col("date_time")))
        .withColumn("hour",      hour(col("date_time")))
        .withColumn("dayofweek", dayofweek(col("date_time"))))

    # 4. Tidy strings.
    string_cols = [f.name for f in df.schema.fields if isinstance(f.dataType, StringType)]
    for c in string_cols:
        df = df.withColumn(c, trim(col(c)))
        df = df.withColumn(c, regexp_replace(col(c), r"\s+", " "))
        df = df.fillna({c: "Unknown"})

    # 5. Median-fill weather, zero-fill precipitation, require a target.
    median_temp   = (df.approxQuantile("temp",       [0.5], 0.01) or [0.0])[0]
    median_clouds = (df.approxQuantile("clouds_all", [0.5], 0.01) or [0])[0]
    df = df.fillna({"temp": median_temp, "clouds_all": median_clouds})
    df = df.fillna({"rain_1h": 0.0, "snow_1h": 0.0})
    df = df.dropna(subset=["traffic_volume"])
    return df

spark = (SparkSession.builder
    .appName("NovaStad_Pipeline_Local")
    .master("local[*]")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.memory", "4g")       
    .getOrCreate())

df_train_raw = (spark.read
    .schema(metro_schema)
    .option("header", True)
    .csv(TRAIN_DATA_PATH))

if df_train_raw.rdd.isEmpty():
    raise RuntimeError(f"No rows loaded from {TRAIN_DATA_PATH}")

df_train = clean_cloud_data(df_train_raw)
print(f"Rows after cleaning: {df_train.count()}")
df_train.printSchema()

26/06/21 01:37:11 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , holiday, temp, rain_1h, snow_1h, clouds_all, weather_main, weather_description, date_time, year, month, day, hour, traffic_volume
 Schema: _c0, holiday, temp, rain_1h, snow_1h, clouds_all, weather_main, weather_description, date_time, year, month, day, hour, traffic_volume
Expected: _c0 but found: 
CSV file: file:///Users/thidaratchuram/Documents/GitHub/MLops-City/Metro_Interstate_Traffic_Volume_train.csv


Rows after cleaning: 40239
root
 |-- holiday: string (nullable = false)
 |-- temp: double (nullable = false)
 |-- rain_1h: double (nullable = false)
 |-- snow_1h: double (nullable = false)
 |-- clouds_all: integer (nullable = true)
 |-- weather_main: string (nullable = false)
 |-- weather_description: string (nullable = false)
 |-- date_time: timestamp (nullable = true)
 |-- traffic_volume: integer (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- day: integer (nullable = true)
 |-- hour: integer (nullable = true)
 |-- dayofweek: integer (nullable = true)



In [14]:
# Stel lokale MLflow tracking in
MLFLOW_TRACKING_URI = "sqlite:///mlflow.db"

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment("Cloud_Traffic_Forecasting")

feature_cols = ["temp", "rain_1h", "snow_1h", "clouds_all", "month", "hour", "dayofweek"]
assembler = VectorAssembler(inputCols=feature_cols, outputCol="raw_features", handleInvalid="skip")
scaler = StandardScaler(inputCol="raw_features", outputCol="features", withStd=True, withMean=False)

rf = RandomForestRegressor(featuresCol="features", labelCol="traffic_volume", seed=42)
cloud_pipeline = Pipeline(stages=[assembler, scaler, rf])

paramGrid = (ParamGridBuilder()
             .addGrid(rf.numTrees, [20, 50])
             .addGrid(rf.maxDepth, [5, 10])
             .build())

evaluator = RegressionEvaluator(labelCol="traffic_volume", predictionCol="prediction", metricName="rmse")

cv = CrossValidator(estimator=cloud_pipeline, estimatorParamMaps=paramGrid, evaluator=evaluator, numFolds=3)


df_test = clean_cloud_data(
    spark.read.schema(metro_schema).option("header", True).csv(TEST_DATA_PATH)
)


with mlflow.start_run(run_name="RandomForest_Tuning_Local"):
    print("Start training & tuning cloud model. Dit kan even duren...")
    
    cv_model = cv.fit(df_train)
    best_model = cv_model.bestModel
    predictions = best_model.transform(df_test)
    
    rmse = evaluator.evaluate(predictions)
    mae = RegressionEvaluator(labelCol="traffic_volume", predictionCol="prediction", metricName="mae").evaluate(predictions)
    
    best_rf = best_model.stages[-1]
    mlflow.log_param("best_numTrees", best_rf.getNumTrees)
    mlflow.log_param("best_maxDepth", best_rf.getOrDefault('maxDepth'))
    mlflow.log_metric("RMSE", rmse)
    mlflow.log_metric("MAE", mae)
    
    print(f"Training Cloud Model voltooid, RMSE: {rmse:.2f}, MAE: {mae:.2f}")

Start training & tuning cloud model. Dit kan even duren...


26/06/21 01:37:24 WARN DAGScheduler: Broadcasting large task binary with size 1437.1 KiB
26/06/21 01:37:25 WARN DAGScheduler: Broadcasting large task binary with size 2.3 MiB
26/06/21 01:37:36 WARN DAGScheduler: Broadcasting large task binary with size 1093.6 KiB
26/06/21 01:37:38 WARN DAGScheduler: Broadcasting large task binary with size 1966.9 KiB
26/06/21 01:37:39 WARN DAGScheduler: Broadcasting large task binary with size 3.4 MiB
26/06/21 01:37:42 WARN DAGScheduler: Broadcasting large task binary with size 5.7 MiB
26/06/21 01:37:45 WARN DAGScheduler: Broadcasting large task binary with size 1331.4 KiB
26/06/21 01:37:56 WARN DAGScheduler: Broadcasting large task binary with size 1419.4 KiB
26/06/21 01:37:58 WARN DAGScheduler: Broadcasting large task binary with size 2.3 MiB
26/06/21 01:38:08 WARN DAGScheduler: Broadcasting large task binary with size 1098.1 KiB
26/06/21 01:38:09 WARN DAGScheduler: Broadcasting large task binary with size 1973.3 KiB
26/06/21 01:38:11 WARN DAGSchedul

Training Cloud Model voltooid, RMSE: 551.14, MAE: 343.48


## Pas de path in je data.yaml bestand voordat je het model train!!!

In [15]:
mlflow.set_experiment("YOLOv8_Traffic_Detection")

with mlflow.start_run(run_name="yolov8n_edge_model_local"):
    
    actual_epochs = 50
    actual_batch = 16
    actual_optimizer = "AdamW"
    
    mlflow.log_param("model", "yolov8n")
    mlflow.log_param("epochs", actual_epochs)
    mlflow.log_param("imgsz", 640)
    mlflow.log_param("batch", actual_batch)
    mlflow.log_param("optimizer", actual_optimizer)

    model = YOLO("yolov8n.pt")

    print("Start YOLO training lokaal...")
    start_time = time.time()

    # Zorg dat je 'data.yaml' ook relatieve lokale paden heeft!
    yolo_results = model.train(
        data="data.yaml",
        epochs=actual_epochs,
        imgsz=640,
        batch=actual_batch,
        optimizer=actual_optimizer,
        workers=0, # Voorkomt crashes in VS code op Windows/Mac!
        project="traffic_yolo",
        name="yolov8n_edge"
    )

    training_time = time.time() - start_time
    metrics = model.val()

    mlflow.log_metric("training_time_seconds", training_time)
    mlflow.log_metric("mAP50", float(metrics.box.map50))



Start YOLO training lokaal...
New https://pypi.org/project/ultralytics/8.4.72 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.67 🚀 Python-3.10.19 torch-2.12.0 CPU (Apple M4 Pro)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov8n

2026/06/21 01:38:56 WARNING mlflow.tracking.fluent: Exception raised while enabling autologging for keras: No module named 'tensorflow.keras'
2026/06/21 01:38:56 WARNING mlflow.tracking.fluent: Exception raised while enabling autologging for tensorflow: No module named 'tensorflow.keras'
2026/06/21 01:38:56 WARNING mlflow.spark: With Pyspark >= 3.2, PYSPARK_PIN_THREAD environment variable must be set to false for Spark datasource autologging to work.
2026/06/21 01:38:56 INFO mlflow.tracking.fluent: Autologging successfully enabled for pyspark.
2026/06/21 01:38:56 INFO mlflow.pyspark.ml: No SparkSession detected. Autologging will log pyspark.ml models contained in the default allowlist. To specify a custom allowlist, initialize a SparkSession prior to calling mlflow.pyspark.ml.autolog() and specify the path to your allowlist file via the spark.mlflow.pysparkml.autolog.logModelAllowlistFile conf.
2026/06/21 01:38:56 INFO mlflow.tracking.fluent: Autologging successfully enabled for pyspar

MLflow: logging run_id(df15f8e18c024b92bd9335929c4cc35a) to sqlite:///mlflow.db
MLflow: disable with 'yolo settings mlflow=False'
WARNING ⚠️ MLflow: Failed to initialize: Changing param values is not allowed. Params were already logged='[{'key': 'model', 'old_value': 'yolov8n', 'new_value': 'yolov8n.pt'}]' for run ID='df15f8e18c024b92bd9335929c4cc35a'.
WARNING ⚠️ MLflow: Not tracking this run
Image sizes 640 train, 640 val
Using 0 dataloader workers
Logging results to /Users/thidaratchuram/Documents/GitHub/MLops-City/runs/detect/traffic_yolo/yolov8n_edge-12
Starting training for 50 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       1/50         0G      1.841      1.728      1.167        341        640: 100% ━━━━━━━━━━━━ 155/155 3.3s/it 8:252.5ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 16/16 2.0s/it 31.5s2.0ss
                   all        500      10165      0.454      0.162

In [16]:
import os

# --- TFLITE EXPORT VOOR EDGE DEPLOYMENT ---
from ultralytics import YOLO
best_model = YOLO("runs/detect/traffic_yolo/yolov8n_edge-11/weights/best.pt")

## Export met INT8 Quantization voor <10MB requirement
export_path = best_model.export(format="tflite", half=True, data="data.yaml")

if os.path.exists(export_path):
    tflite_size_mb = os.path.getsize(export_path) / (1024 * 1024)
    mlflow.log_metric("tflite_model_size_mb", tflite_size_mb)
    mlflow.log_artifact(export_path, artifact_path="yolo_tflite_edge_model")
    print(f"TFLite export succesvol. Modelgrootte: {tflite_size_mb:.2f} MB")


Ultralytics 8.4.67 🚀 Python-3.10.19 torch-2.12.0 CPU (Apple M4 Pro)
Model summary (fused): 73 layers, 3,007,208 parameters, 0 gradients, 8.1 GFLOPs

PyTorch: starting from 'runs/detect/traffic_yolo/yolov8n_edge-11/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 12, 8400) (5.9 MB)

ONNX: starting export with onnx 1.22.0 opset 20...
ONNX: slimming with onnxslim 0.1.94...
ONNX: export success ✅ 0.4s, saved as 'runs/detect/traffic_yolo/yolov8n_edge-11/weights/best.onnx' (11.8 MB)

TensorFlow SavedModel: starting export with tensorflow 2.15.0...
TensorFlow SavedModel: starting TFLite export with onnx2tf 1.28.8...
Saved artifact at 'runs/detect/traffic_yolo/yolov8n_edge-11/weights/best_saved_model'. The following endpoints are available:

* Endpoint 'serving_default'
  inputs_0 (POSITIONAL_ONLY): TensorSpec(shape=(1, 640, 640, 3), dtype=tf.float32, name='images')
Output Type:
  TensorSpec(shape=(1, 12, 8400), dtype=tf.float32, name=None)
Captures:
  1379141704

Summary on the non-converted ops:
---------------------------------
 * Accepted dialects: tfl, builtin, func
 * Non-Converted Ops: 156, Total Ops 411, % non-converted = 37.96 %
 * 156 ARITH ops

- arith.constant:  156 occurrences  (f32: 131, i32: 25)



  (f32: 8)
  (f32: 16)
  (f32: 64)
  (f32: 57)
  (f32: 3)
  (f32: 58)
  (f32: 7)
  (f32: 9)
  (f32: 2)
  (f32: 1)
  (f32: 18)
  (f32: 2)
  (f32: 7)


TensorFlow SavedModel: export success ✅ 7.3s, saved as 'runs/detect/traffic_yolo/yolov8n_edge-11/weights/best_saved_model' (29.5 MB)

TensorFlow Lite: starting export with tensorflow 2.15.0...
TensorFlow Lite: export success ✅ 0.0s, saved as 'runs/detect/traffic_yolo/yolov8n_edge-11/weights/best_saved_model/best_float16.tflite' (5.9 MB)

Export complete (7.4s)
Results saved to /Users/thidaratchuram/Documents/GitHub/MLops-City/runs/detect/traffic_yolo/yolov8n_edge-11/weights/best_saved_model/best_float16.tflite
Predict:         yolo predict task=detect model=runs/detect/traffic_yolo/yolov8n_edge-11/weights/best_saved_model/best_float16.tflite imgsz=640 half
Validate:        yolo val task=detect model=runs/detect/traffic_yolo/yolov8n_edge-11/weights/best_saved_model/best_float16.tflite imgsz=640 data=data.yaml half 
Visualize:       https://netron.app
TFLite export succesvol. Modelgrootte: 5.91 MB


Summary on the non-converted ops:
---------------------------------
 * Accepted dialects: tfl, builtin, func
 * Non-Converted Ops: 156, Total Ops 542, % non-converted = 28.78 %
 * 156 ARITH ops

- arith.constant:  156 occurrences  (f16: 131, i32: 25)



  (f32: 8)
  (f32: 16)
  (f32: 64)
  (f32: 131)
  (f32: 57)
  (f32: 3)
  (f32: 58)
  (f32: 7)
  (f32: 9)
  (f32: 2)
  (f32: 1)
  (f32: 18)
  (f32: 2)
  (f32: 7)


Maak in VS Code een nieuwe map src/api/ en maak daarin een bestand genaamd main.py.

In [0]:
from fastapi import FastAPI
from pydantic import BaseModel
from pyspark.sql import SparkSession
from pyspark.ml import PipelineModel
import pandas as pd

# 1. Definieer de API
app = FastAPI(
    title="Nova Stad Traffic Voorspeller",
    description="API voor het voorspellen van verkeersdrukte o.b.v. weerdata."
)

# 2. Definieer hoe de input data eruit moet zien
class TrafficRequest(BaseModel):
    temp: float
    rain_1h: float
    snow_1h: float
    clouds_all: int
    month: int
    hour: int
    dayofweek: int

# We starten Spark en laden het model zodra de API opstart
spark = SparkSession.builder.master("local[1]").appName("TrafficAPI").getOrCreate()

# LET OP: In de praktijk laad je hier je opgeslagen model in:
# model = PipelineModel.load("../../models/cloud_hybrid/random_forest_model")

@app.get("/")
def home():
    return {"message": "De Nova Stad Traffic API is online! Ga naar /docs voor de interface."}

@app.post("/predict")
def predict_traffic(data: TrafficRequest):
    # Converteer de inkomende JSON request naar een Pandas DataFrame en dan naar Spark
    pdf = pd.DataFrame([data.dict()])
    sdf = spark.createDataFrame(pdf)
    
    # In productie doe je hier: prediction = model.transform(sdf).collect()[0]["prediction"]
    # Voor nu simuleren we een antwoord:
    gesimuleerde_voorspelling = 3500 
    
    return {
        "status": "success",
        "verwachte_verkeersvolume": gesimuleerde_voorspelling,
        "gebruikte_features": data.dict()
    }

Maak in de allerhoogste map van je project (naast je README.md) een bestand genaamd Dockerfile (geen extensie!).

In [0]:
# Gebruik een lichte Python basis-image
FROM python:3.9-slim

# Omdat we PySpark gebruiken, hebben we Java nodig in de container
RUN apt-get update && \
    apt-get install -y default-jre && \
    apt-get clean

# Stel de werkmap in
WORKDIR /app

# Kopieer de benodigdheden en installeer deze
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Kopieer de source code en eventuele modellen
COPY src/ /app/src/

# Open poort 8000 voor de API
EXPOSE 8000

# Start de FastAPI server wanneer de container start
CMD ["uvicorn", "src.api.main:app", "--host", "0.0.0.0", "--port", "8000"]

We gaan GitHub zo instellen dat hij bij elke push automatisch checkt of je code netjes is (volgens PEP8) en of het werkt.
Maak de mappen .github/workflows/ aan en maak daarin het bestand ci_cd.yml.

In [0]:
name: Nova Stad MLOps Pipeline (CI/CD)

on:
  push:
    branches: [ "main" ]

jobs:
  build_and_test:
    runs-on: ubuntu-latest

    steps:
    - name: Code ophalen
      uses: actions/checkout@v3

    - name: Python instellen
      uses: actions/setup-python@v4
      with:
        python-version: "3.9"

    - name: Dependencies installeren
      run: |
        python -m pip install --upgrade pip
        pip install flake8 pytest
        if [ -f requirements.txt ]; then pip install -r requirements.txt; fi

    - name: Linting met Flake8 (Check PEP8)
      run: |
        # Stop de build als er ernstige syntax fouten zijn
        flake8 src/ --count --select=E9,F63,F7,F82 --show-source --statistics

"Monitoren en Drift detectie". Hiervoor gaan we terug naar je Jupyter Notebook. Voeg aan het einde van je notebook deze 2 cellen toe. Installeer eerst even het package via je terminal: pip install evidently.

## 6. Monitoren & Data Drift Detectie (MLOps)
Om te voldoen aan de monitoring-eisen van een productie-systeem, moeten we detecteren of de data over tijd verandert (Data Drift). Als het bijvoorbeeld in de test-dataset plotseling veel meer regent dan in de train-dataset, zal ons model slechter presteren.

We gebruiken **Evidently AI** om een geautomatiseerd drift-dashboard te genereren. Indien er significante drift wordt gedetecteerd in kritieke variabelen (zoals `temp` of `rain_1h`), dient dit als trigger om de Continuous Training (CT) pijplijn te activeren.

In [0]:
import pandas as pd
from evidently.report import Report
from evidently.metric_preset import DataDriftPreset

print("Start Data Drift Analyse...")

# Evidently AI werkt optimaal met Pandas.
# Om geheugen te besparen converteren we een representatieve sample (10.000 rijen) van Spark naar Pandas.
features_to_monitor = ["temp", "rain_1h", "clouds_all", "traffic_volume"]

train_sample_pd = df_train.select(features_to_monitor).limit(10000).toPandas()
test_sample_pd = df_test.select(features_to_monitor).limit(10000).toPandas()

# Genereer het Data Drift Report
drift_report = Report(metrics=[DataDriftPreset()])
drift_report.run(reference_data=train_sample_pd, current_data=test_sample_pd)

# Optie 1: Sla het dashboard op als een interactieve HTML pagina (zeer professioneel voor MLOps logging!)
drift_report.save_html("data_drift_dashboard.html")
print("Dashboard succesvol opgeslagen als 'data_drift_dashboard.html'")

# Optie 2: Toon het direct in dit notebook (Werkt perfect in VS Code!)
drift_report.show(mode='inline')